In [1]:
"""
甲状腺癌复发预测 - 10模型对比完整实现（PDF输出版）
修复 SGDClassifier predict_proba 问题，支持PDF输出
"""

# ============================================================
# 第一部分: 环境配置和导入
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                            roc_auc_score, confusion_matrix, classification_report,
                            roc_curve, precision_recall_curve, brier_score_loss, cohen_kappa_score)
from sklearn.calibration import calibration_curve, CalibratedClassifierCV

# 10个模型导入
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier, ExtraTreesClassifier

# 梯度提升库
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not available, will be skipped")

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("LightGBM not available, will be skipped")

try:
    from catboost import CatBoostClassifier
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False
    print("CatBoost not available, will be skipped")

import warnings
import os
import joblib
from datetime import datetime
warnings.filterwarnings('ignore')

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

# 设置随机种子保证可重复性
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("=" * 70)
print("甲状腺癌复发预测 - 10模型对比分析系统（PDF输出版）")
print("=" * 70)

# ============================================================
# 第二部分: 数据加载和预处理
# ============================================================

def load_and_preprocess_data(file_path=None):
    """
    加载和预处理甲状腺癌数据
    """
    print("\n" + "=" * 70)
    print("步骤 1: 数据加载和预处理")
    print("=" * 70)

    if file_path and os.path.exists(file_path):
        df = pd.read_csv(file_path)
        print(f"从 {file_path} 加载数据")
    else:
        print("创建模拟数据集（基于论文描述的383例样本）")
        df = create_synthetic_data()

    print(f"总样本数: {len(df)}")
    print(f"特征数: {df.shape[1] - 1}")
    print(f"\n类别分布:")
    print(df['Recurred'].value_counts())
    print(f"\n复发率: {df['Recurred'].mean():.2%}")

    return df

def create_synthetic_data(n_samples=383, random_state=42):
    """
    基于论文统计特征创建模拟数据
    """
    np.random.seed(random_state)

    n_recurred = int(n_samples * 0.282)
    n_not_recurred = n_samples - n_recurred

    def create_class_samples(n, recurred=True):
        data = {}
        data['Age'] = np.clip(np.random.normal(52 if recurred else 42, 12, n).astype(int), 18, 85)
        data['Gender'] = np.random.choice([0, 1], n, p=[0.65, 0.35])
        data['Smoking'] = np.random.choice([0, 1], n, p=[0.85, 0.15])
        data['Hx Smoking'] = np.random.choice([0, 1], n, p=[0.75, 0.25])
        data['Hx Radiothreapy'] = np.random.choice([0, 1], n, p=[0.90, 0.10])
        data['Thyroid Function'] = np.random.choice([0, 1, 2, 3, 4], n, p=[0.05, 0.10, 0.70, 0.10, 0.05])
        data['Physical Examination'] = np.random.choice([0, 1, 2, 3, 4], n, p=[0.15, 0.25, 0.30, 0.15, 0.15])

        if recurred:
            data['Adenopathy'] = np.random.choice([0, 1, 2, 3, 4, 5], n, p=[0.20, 0.15, 0.15, 0.30, 0.10, 0.10])
            data['Risk'] = np.random.choice([0, 1, 2], n, p=[0.20, 0.40, 0.40])
            data['Response'] = np.random.choice([0, 1, 2, 3], n, p=[0.15, 0.25, 0.40, 0.20])
        else:
            data['Adenopathy'] = np.random.choice([0, 1, 2, 3, 4, 5], n, p=[0.10, 0.05, 0.05, 0.70, 0.05, 0.05])
            data['Risk'] = np.random.choice([0, 1, 2], n, p=[0.60, 0.30, 0.10])
            data['Response'] = np.random.choice([0, 1, 2, 3], n, p=[0.70, 0.20, 0.08, 0.02])

        data['Pathology'] = np.random.choice([0, 1, 2, 3], n, p=[0.15, 0.10, 0.25, 0.50])
        data['Focality'] = np.random.choice([0, 1], n, p=[0.40, 0.60])
        data['T'] = np.random.choice([0, 1, 2, 3, 4, 5, 6], n, p=[0.20, 0.20, 0.20, 0.15, 0.15, 0.05, 0.05])
        data['N'] = np.random.choice([0, 1, 2], n, p=[0.60, 0.25, 0.15])
        data['M'] = np.random.choice([0, 1], n, p=[0.95, 0.05])
        data['Stage'] = np.random.choice([0, 1, 2, 3, 4], n, p=[0.50, 0.20, 0.15, 0.10, 0.05])

        data['Recurred'] = 1 if recurred else 0
        return pd.DataFrame(data)

    df_recurred = create_class_samples(n_recurred, recurred=True)
    df_not_recurred = create_class_samples(n_not_recurred, recurred=False)

    df = pd.concat([df_recurred, df_not_recurred], ignore_index=True)
    df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    return df

# ============================================================
# 第三部分: 特征工程和数据保存
# ============================================================

def prepare_features_and_save(df, output_dir='./results'):
    """
    准备特征矩阵和目标变量，并保存数据集
    """
    print("\n" + "=" * 70)
    print("步骤 2: 特征工程和数据集保存")
    print("=" * 70)

    feature_names = ['Age', 'Gender', 'Smoking', 'Hx Smoking', 'Hx Radiothreapy',
                     'Thyroid Function', 'Physical Examination', 'Adenopathy',
                     'Pathology', 'Focality', 'Risk', 'T', 'N', 'M', 'Stage', 'Response']

    X = df[feature_names].copy()
    y = df['Recurred'].copy()

    print(f"特征矩阵形状: {X.shape}")

    # 划分训练集和测试集 (70:30)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
    )

    print(f"\n训练集: {len(X_train)} 样本 (复发: {sum(y_train)}, 未复发: {len(y_train)-sum(y_train)})")
    print(f"测试集: {len(X_test)} 样本 (复发: {sum(y_test)}, 未复发: {len(y_test)-sum(y_test)})")

    # 标准化
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 保存原始数据集
    os.makedirs(output_dir, exist_ok=True)

    # 保存完整数据集
    df.to_csv(f'{output_dir}/complete_dataset.csv', index=False)
    print(f"\n✓ 保存完整数据集: complete_dataset.csv ({len(df)} 样本)")

    # 保存训练集（包含特征和目标）
    train_df = X_train.copy()
    train_df['Recurred'] = y_train.values
    train_df.to_csv(f'{output_dir}/training_set.csv', index=False)
    print(f"✓ 保存训练集: training_set.csv ({len(train_df)} 样本)")

    # 保存测试集（包含特征和目标）
    test_df = X_test.copy()
    test_df['Recurred'] = y_test.values
    test_df.to_csv(f'{output_dir}/testing_set.csv', index=False)
    print(f"✓ 保存测试集: testing_set.csv ({len(test_df)} 样本)")

    # 保存标准化后的数据（用于需要标准化的模型）
    train_scaled_df = pd.DataFrame(X_train_scaled, columns=feature_names)
    train_scaled_df['Recurred'] = y_train.values
    train_scaled_df.to_csv(f'{output_dir}/training_set_scaled.csv', index=False)
    print(f"✓ 保存标准化训练集: training_set_scaled.csv")

    test_scaled_df = pd.DataFrame(X_test_scaled, columns=feature_names)
    test_scaled_df['Recurred'] = y_test.values
    test_scaled_df.to_csv(f'{output_dir}/testing_set_scaled.csv', index=False)
    print(f"✓ 保存标准化测试集: testing_set_scaled.csv")

    # 保存数据集信息
    dataset_info = {
        'total_samples': len(df),
        'n_features': len(feature_names),
        'training_samples': len(X_train),
        'testing_samples': len(X_test),
        'recurred_train': int(sum(y_train)),
        'not_recurred_train': int(len(y_train) - sum(y_train)),
        'recurred_test': int(sum(y_test)),
        'not_recurred_test': int(len(y_test) - sum(y_test)),
        'feature_names': feature_names,
        'random_state': RANDOM_STATE,
        'test_size': 0.3,
        'creation_time': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }

    with open(f'{output_dir}/dataset_info.txt', 'w') as f:
        for key, value in dataset_info.items():
            f.write(f"{key}: {value}\n")
    print(f"✓ 保存数据集信息: dataset_info.txt")

    # 保存scaler
    joblib.dump(scaler, f'{output_dir}/scaler.pkl')
    print(f"✓ 保存标准化器: scaler.pkl")

    return X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled, feature_names, scaler

# ============================================================
# 第四部分: 模型定义和超参数配置
# ============================================================

def get_models_and_params():
    """
    定义10个模型及其超参数搜索空间
    """
    print("\n" + "=" * 70)
    print("步骤 3: 模型定义")
    print("=" * 70)

    models = {}

    # 1. Logistic Regression
    models['Logistic Regression'] = {
        'model': LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
        'params': {
            'C': [0.01, 0.1, 1, 10, 100],
            'penalty': ['l1', 'l2'],
            'solver': ['liblinear', 'saga']
        },
        'scale': True,
        'needs_calibration': False
    }

    # 2. Decision Tree
    models['Decision Tree'] = {
        'model': DecisionTreeClassifier(random_state=RANDOM_STATE),
        'params': {
            'criterion': ['gini', 'entropy'],
            'max_depth': [3, 5, 7, 10, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        },
        'scale': False,
        'needs_calibration': False
    }

    # 3. SVM
    models['SVM'] = {
        'model': SVC(random_state=RANDOM_STATE, probability=True),
        'params': {
            'C': [0.1, 1, 10],
            'kernel': ['rbf', 'linear'],
            'gamma': ['scale', 'auto']
        },
        'scale': True,
        'needs_calibration': False
    }

    # 4. KNN
    models['KNN'] = {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7, 9, 11],
            'weights': ['uniform', 'distance'],
            'metric': ['euclidean', 'manhattan']
        },
        'scale': True,
        'needs_calibration': False
    }

    # 5. Naive Bayes
    models['Naive Bayes'] = {
        'model': GaussianNB(),
        'params': {},
        'scale': False,
        'needs_calibration': False
    }

    # 6. Random Forest
    models['Random Forest'] = {
        'model': RandomForestClassifier(random_state=RANDOM_STATE),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5],
            'min_samples_leaf': [1, 2]
        },
        'scale': False,
        'needs_calibration': False
    }

    # # 7. Gradient Boosting
    # models['Gradient Boosting'] = {
    #     'model': GradientBoostingClassifier(random_state=RANDOM_STATE),
    #     'params': {
    #         'n_estimators': [100, 200],
    #         'learning_rate': [0.05, 0.1],
    #         'max_depth': [3, 5]
    #     },
    #     'scale': False,
    #     'needs_calibration': False
    # }

    # 8. XGBoost
    if XGBOOST_AVAILABLE:
        models['XGBoost'] = {
            'model': XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss'),
            'params': {
                'n_estimators': [100, 200],
                'max_depth': [3, 5, 7],
                'learning_rate': [0.05, 0.1],
                'subsample': [0.8, 1.0]
            },
            'scale': False,
            'needs_calibration': False
        }

    # 9. LightGBM
    if LIGHTGBM_AVAILABLE:
        models['LightGBM'] = {
            'model': LGBMClassifier(random_state=RANDOM_STATE, verbose=-1),
            'params': {
                'n_estimators': [100, 200],
                'max_depth': [3, 5, 7],
                'learning_rate': [0.05, 0.1],
                'num_leaves': [31, 50]
            },
            'scale': False,
            'needs_calibration': False
        }

    # 10. CatBoost
    if CATBOOST_AVAILABLE:
        models['CatBoost'] = {
            'model': CatBoostClassifier(random_state=RANDOM_STATE, verbose=False),
            'params': {
                'iterations': [100, 200],
                'depth': [3, 5, 7],
                'learning_rate': [0.05, 0.1]
            },
            'scale': False,
            'needs_calibration': False
        }

    print(f"共定义 {len(models)} 个模型:")
    for name in models.keys():
        print(f"  - {name}")

    return models

# ============================================================
# 第五部分: 模型训练和优化
# ============================================================

def get_predict_proba(model, X):
    """
    安全地获取预测概率
    """
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    elif hasattr(model, 'decision_function'):
        from scipy.special import expit
        return expit(model.decision_function(X))
    else:
        raise AttributeError("Model has no predict_proba or decision_function")

def train_and_optimize_models(models, X_train, X_test, y_train, y_test,
                              X_train_scaled, X_test_scaled):
    """
    训练和优化所有模型，同时计算训练集和测试集的预测结果
    """
    print("\n" + "=" * 70)
    print("步骤 4: 模型训练和超参数优化")
    print("=" * 70)

    results = {}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    for name, config in models.items():
        print(f"\n{'='*60}")
        print(f"训练模型: {name}")
        print(f"{'='*60}")

        if config['scale']:
            X_tr = X_train_scaled
            X_te = X_test_scaled
            print("使用标准化数据")
        else:
            X_tr = X_train.values
            X_te = X_test.values
            print("使用原始数据")

        # 超参数优化
        if config['params']:
            print("执行GridSearchCV...")
            grid = GridSearchCV(
                config['model'],
                config['params'],
                cv=cv,
                scoring='roc_auc',
                n_jobs=-1,
                verbose=0
            )
            grid.fit(X_tr, y_train)
            best_model = grid.best_estimator_
            print(f"最优参数: {grid.best_params_}")
            print(f"最优CV AUC: {grid.best_score_:.4f}")
        else:
            best_model = config['model']
            best_model.fit(X_tr, y_train)
            cv_scores = cross_val_score(best_model, X_tr, y_train, cv=cv, scoring='roc_auc')
            print(f"CV AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

        # 测试集预测
        y_pred = best_model.predict(X_te)

        try:
            y_prob = get_predict_proba(best_model, X_te)
        except Exception as e:
            print(f"警告: 使用校准: {e}")
            calibrated = CalibratedClassifierCV(best_model, cv=5, method='sigmoid')
            calibrated.fit(X_tr, y_train)
            y_prob = calibrated.predict_proba(X_te)[:, 1]
            best_model = calibrated

        metrics_test = calculate_metrics(y_test, y_pred, y_prob)

        # 训练集预测
        y_pred_train = best_model.predict(X_tr)
        try:
            y_prob_train = get_predict_proba(best_model, X_tr)
        except:
            y_prob_train = best_model.predict_proba(X_tr)[:, 1] if hasattr(best_model, 'predict_proba') else None

        metrics_train = calculate_metrics(y_train, y_pred_train, y_prob_train)

        print(f"\n训练集性能:")
        print(f"  Accuracy: {metrics_train['Accuracy']:.4f} (95% CI: {metrics_train['Accuracy_CI'][0]:.3f}-{metrics_train['Accuracy_CI'][1]:.3f})")
        print(f"  Precision: {metrics_train['Precision']:.4f} (95% CI: {metrics_train['Precision_CI'][0]:.3f}-{metrics_train['Precision_CI'][1]:.3f})")
        print(f"  Recall: {metrics_train['Recall']:.4f} (95% CI: {metrics_train['Recall_CI'][0]:.3f}-{metrics_train['Recall_CI'][1]:.3f})")
        print(f"  F1-score: {metrics_train['F1-score']:.4f} (95% CI: {metrics_train['F1_CI'][0]:.3f}-{metrics_train['F1_CI'][1]:.3f})")
        print(f"  AUC: {metrics_train['AUC']:.4f} (95% CI: {metrics_train['AUC_CI'][0]:.3f}-{metrics_train['AUC_CI'][1]:.3f})")

        print(f"\n测试集性能:")
        print(f"  Accuracy: {metrics_test['Accuracy']:.4f} (95% CI: {metrics_test['Accuracy_CI'][0]:.3f}-{metrics_test['Accuracy_CI'][1]:.3f})")
        print(f"  Precision: {metrics_test['Precision']:.4f} (95% CI: {metrics_test['Precision_CI'][0]:.3f}-{metrics_test['Precision_CI'][1]:.3f})")
        print(f"  Recall: {metrics_test['Recall']:.4f} (95% CI: {metrics_test['Recall_CI'][0]:.3f}-{metrics_test['Recall_CI'][1]:.3f})")
        print(f"  F1-score: {metrics_test['F1-score']:.4f} (95% CI: {metrics_test['F1_CI'][0]:.3f}-{metrics_test['F1_CI'][1]:.3f})")
        print(f"  AUC: {metrics_test['AUC']:.4f} (95% CI: {metrics_test['AUC_CI'][0]:.3f}-{metrics_test['AUC_CI'][1]:.3f})")

        results[name] = {
            'model': best_model,
            'metrics': metrics_test,
            'metrics_train': metrics_train,
            'y_pred': y_pred,
            'y_prob': y_prob,
            'y_pred_train': y_pred_train,
            'y_prob_train': y_prob_train,
            'use_scaled': config['scale']
        }

    return results

from sklearn.utils import resample

def compute_confidence_interval(y_true, y_pred, y_prob, metric_func, n_bootstrap=1000, confidence=0.95):
    """
    使用Bootstrap计算指标的95%置信区间
    """
    rng = np.random.RandomState(RANDOM_STATE)
    bootstrapped_scores = []

    for i in range(n_bootstrap):
        indices = rng.randint(0, len(y_true), len(y_true))
        if len(np.unique(y_true.iloc[indices] if hasattr(y_true, 'iloc') else y_true[indices])) < 2:
            continue
        score = metric_func(y_true.iloc[indices] if hasattr(y_true, 'iloc') else y_true[indices],
                          y_pred.iloc[indices] if hasattr(y_pred, 'iloc') else y_pred[indices])
        bootstrapped_scores.append(score)

    if len(bootstrapped_scores) == 0:
        return (0, 0)

    alpha = (1 - confidence) / 2
    ci_lower = np.percentile(bootstrapped_scores, alpha * 100)
    ci_upper = np.percentile(bootstrapped_scores, (1 - alpha) * 100)
    return (ci_lower, ci_upper)

def compute_auc_ci(y_true, y_prob, n_bootstrap=1000, confidence=0.95):
    """
    使用Bootstrap计算AUC的95%置信区间
    """
    rng = np.random.RandomState(RANDOM_STATE)
    bootstrapped_scores = []

    for i in range(n_bootstrap):
        indices = rng.randint(0, len(y_true), len(y_true))
        if len(np.unique(y_true.iloc[indices] if hasattr(y_true, 'iloc') else y_true[indices])) < 2:
            continue
        score = roc_auc_score(y_true.iloc[indices] if hasattr(y_true, 'iloc') else y_true[indices],
                             y_prob[indices])
        bootstrapped_scores.append(score)

    if len(bootstrapped_scores) == 0:
        return (0, 0)

    alpha = (1 - confidence) / 2
    ci_lower = np.percentile(bootstrapped_scores, alpha * 100)
    ci_upper = np.percentile(bootstrapped_scores, (1 - alpha) * 100)
    return (ci_lower, ci_upper)

def calculate_metrics(y_true, y_pred, y_prob, compute_ci=True):
    """
    计算所有评估指标，并可选计算95%置信区间
    """
    metrics = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Accuracy_CI': (0, 0),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Precision_CI': (0, 0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'Recall_CI': (0, 0),
        'F1-score': f1_score(y_true, y_pred, zero_division=0),
        'F1_CI': (0, 0),
        'AUC': roc_auc_score(y_true, y_prob),
        'AUC_CI': (0, 0),
        'Brier Score': brier_score_loss(y_true, y_prob),
        'Brier_CI': (0, 0),
        'AP': 0  # Average Precision
    }

    # Calculate Average Precision for PR curve
    try:
        from sklearn.metrics import average_precision_score
        metrics['AP'] = average_precision_score(y_true, y_prob)
    except:
        pass

    if compute_ci:
        metrics['Accuracy_CI'] = compute_confidence_interval(
            y_true, y_pred, y_prob, accuracy_score)
        metrics['Precision_CI'] = compute_confidence_interval(
            y_true, y_pred, y_prob, lambda yt, yp: precision_score(yt, yp, zero_division=0))
        metrics['Recall_CI'] = compute_confidence_interval(
            y_true, y_pred, y_prob, lambda yt, yp: recall_score(yt, yp, zero_division=0))
        metrics['F1_CI'] = compute_confidence_interval(
            y_true, y_pred, y_prob, lambda yt, yp: f1_score(yt, yp, zero_division=0))
        metrics['AUC_CI'] = compute_auc_ci(y_true, y_prob)
        metrics['Brier_CI'] = compute_confidence_interval(
            y_true, y_pred, y_prob, brier_score_loss)

    return metrics

# ============================================================
# 第六部分: 堆叠集成模型
# ============================================================

def build_stacking_model(results, X_train, X_test, y_train, y_test,
                        X_train_scaled, X_test_scaled):
    """
    构建堆叠集成模型（支持训练集和测试集评估）
    """
    print("\n" + "=" * 70)
    print("步骤 5: 构建堆叠集成模型")
    print("=" * 70)

    # 使用log loss确保有predict_proba
    base_learners = [
        ('sgd', SGDClassifier(
            loss='log_loss',
            penalty='l1',
            alpha=0.0001,
            max_iter=50,
            random_state=RANDOM_STATE
        )),
        ('et', ExtraTreesClassifier(
            criterion='entropy',
            max_depth=5,
            min_samples_split=10,
            min_samples_leaf=1,
            random_state=RANDOM_STATE
        )),
        ('dt', DecisionTreeClassifier(
            criterion='entropy',
            max_depth=3,
            min_samples_split=4,
            min_samples_leaf=2,
            random_state=RANDOM_STATE
        ))
    ]

    print("训练基学习器...")
    X_tr = X_train.values
    X_te = X_test.values

    for name, model in base_learners:
        model.fit(X_tr, y_train)
        y_prob = model.predict_proba(X_te)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
        print(f"  {name}: AUC = {auc:.4f}")

    # 元学习器
    if XGBOOST_AVAILABLE:
        meta_learner = XGBClassifier(
            n_estimators=150, max_depth=3, learning_rate=0.05, subsample=0.6,
            random_state=RANDOM_STATE, eval_metric='logloss'
        )
    else:
        meta_learner = GradientBoostingClassifier(
            n_estimators=150, max_depth=3, learning_rate=0.05, subsample=0.6,
            random_state=RANDOM_STATE
        )

    stacking = StackingClassifier(
        estimators=base_learners,
        final_estimator=meta_learner,
        cv=5,
        stack_method='predict_proba',
        n_jobs=-1
    )

    print("\n训练堆叠模型...")
    stacking.fit(X_tr, y_train)

    # 测试集预测
    y_pred = stacking.predict(X_te)
    y_prob = stacking.predict_proba(X_te)[:, 1]
    metrics_test = calculate_metrics(y_test, y_pred, y_prob)

    # 训练集预测
    y_pred_train = stacking.predict(X_tr)
    y_prob_train = stacking.predict_proba(X_tr)[:, 1]
    metrics_train = calculate_metrics(y_train, y_pred_train, y_prob_train)

    print(f"\n堆叠模型训练集性能:")
    print(f"  Accuracy: {metrics_train['Accuracy']:.4f} (95% CI: {metrics_train['Accuracy_CI'][0]:.3f}-{metrics_train['Accuracy_CI'][1]:.3f})")
    print(f"  AUC: {metrics_train['AUC']:.4f} (95% CI: {metrics_train['AUC_CI'][0]:.3f}-{metrics_train['AUC_CI'][1]:.3f})")

    print(f"\n堆叠模型测试集性能:")
    print(f"  Accuracy: {metrics_test['Accuracy']:.4f} (95% CI: {metrics_test['Accuracy_CI'][0]:.3f}-{metrics_test['Accuracy_CI'][1]:.3f})")
    print(f"  AUC: {metrics_test['AUC']:.4f} (95% CI: {metrics_test['AUC_CI'][0]:.3f}-{metrics_test['AUC_CI'][1]:.3f})")

    results['Stacking (Paper Method)'] = {
        'model': stacking,
        'metrics': metrics_test,
        'metrics_train': metrics_train,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'y_pred_train': y_pred_train,
        'y_prob_train': y_prob_train,
        'use_scaled': False
    }

    # 改进版堆叠
    print("\n构建改进版堆叠模型 (Top-3)...")
    top3 = sorted(results.items(), key=lambda x: x[1]['metrics']['AUC'], reverse=True)[:3]
    print(f"选择: {[name for name, _ in top3 if name != 'Stacking (Paper Method)'][:3]}")

    top3_learners = []
    for name, res in top3:
        if name != 'Stacking (Paper Method)' and len(top3_learners) < 3:
            if name == 'Logistic Regression':
                m = LogisticRegression(C=1, random_state=RANDOM_STATE, max_iter=1000)
            elif name == 'Random Forest':
                m = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)
            elif name == 'XGBoost':
                m = XGBClassifier(n_estimators=100, random_state=RANDOM_STATE, eval_metric='logloss')
            else:
                m = res['model'].__class__(random_state=RANDOM_STATE)

            X_input = X_train_scaled if res['use_scaled'] else X_train.values
            m.fit(X_input, y_train)
            top3_learners.append((name.replace(' ', '_'), m))

    if len(top3_learners) >= 2:
        stacking_v2 = StackingClassifier(
            estimators=top3_learners,
            final_estimator=LogisticRegression(random_state=RANDOM_STATE),
            cv=5,
            stack_method='predict_proba'
        )

        stacking_v2.fit(X_train_scaled, y_train)

        # 测试集预测
        y_pred_v2 = stacking_v2.predict(X_test_scaled)
        y_prob_v2 = stacking_v2.predict_proba(X_test_scaled)[:, 1]
        metrics_v2_test = calculate_metrics(y_test, y_pred_v2, y_prob_v2)

        # 训练集预测
        y_pred_v2_train = stacking_v2.predict(X_train_scaled)
        y_prob_v2_train = stacking_v2.predict_proba(X_train_scaled)[:, 1]
        metrics_v2_train = calculate_metrics(y_train, y_pred_v2_train, y_prob_v2_train)

        print(f"\n改进版堆叠模型训练集性能:")
        print(f"  Accuracy: {metrics_v2_train['Accuracy']:.4f} (95% CI: {metrics_v2_train['Accuracy_CI'][0]:.3f}-{metrics_v2_train['Accuracy_CI'][1]:.3f})")
        print(f"  AUC: {metrics_v2_train['AUC']:.4f} (95% CI: {metrics_v2_train['AUC_CI'][0]:.3f}-{metrics_v2_train['AUC_CI'][1]:.3f})")

        print(f"\n改进版堆叠模型测试集性能:")
        print(f"  Accuracy: {metrics_v2_test['Accuracy']:.4f} (95% CI: {metrics_v2_test['Accuracy_CI'][0]:.3f}-{metrics_v2_test['Accuracy_CI'][1]:.3f})")
        print(f"  AUC: {metrics_v2_test['AUC']:.4f} (95% CI: {metrics_v2_test['AUC_CI'][0]:.3f}-{metrics_v2_test['AUC_CI'][1]:.3f})")

        results['Stacking v2 (Top3)'] = {
            'model': stacking_v2,
            'metrics': metrics_v2_test,
            'metrics_train': metrics_v2_train,
            'y_pred': y_pred_v2,
            'y_prob': y_prob_v2,
            'y_pred_train': y_pred_v2_train,
            'y_prob_train': y_prob_v2_train,
            'use_scaled': True
        }

    return results

# ============================================================
# 第七部分: 可视化（支持PDF和PNG）
# ============================================================

def save_figure(fig, base_path, formats=['png', 'pdf'], dpi=300):
    """
    保存图为多种格式
    """
    for fmt in formats:
        full_path = f"{base_path}.{fmt}"
        if fmt == 'png':
            fig.savefig(full_path, dpi=dpi, bbox_inches='tight')
        else:  # pdf
            fig.savefig(full_path, bbox_inches='tight', format='pdf')
        print(f"  ✓ {full_path}")

def create_visualizations(results, X_train, X_test, y_train, y_test, feature_names, output_dir='./results'):
    """
    创建所有可视化图表（PNG和PDF），分别绘制训练集和测试集
    """
    print("\n" + "=" * 70)
    print("步骤 6: 生成可视化图表 (PNG + PDF)")
    print("=" * 70)

    os.makedirs(output_dir, exist_ok=True)

    # 1. 性能对比（包含训练集和测试集）
    plot_performance_comparison(results, f'{output_dir}/01_performance_comparison')

    # 2. ROC曲线 - 训练集
    plot_roc_curves(results, y_train, f'{output_dir}/02_roc_curves_train', dataset_type='Train')
    # 2. ROC曲线 - 测试集
    plot_roc_curves(results, y_test, f'{output_dir}/02_roc_curves_test', dataset_type='Test')

    # 3. 校准曲线 - 训练集
    plot_calibration_curves(results, y_train, f'{output_dir}/03_calibration_curves_train', dataset_type='Train')
    # 3. 校准曲线 - 测试集
    plot_calibration_curves(results, y_test, f'{output_dir}/03_calibration_curves_test', dataset_type='Test')

    # 4. 混淆矩阵 - 训练集
    plot_confusion_matrices(results, y_train, f'{output_dir}/04_confusion_matrices_train', dataset_type='Train')
    # 4. 混淆矩阵 - 测试集
    plot_confusion_matrices(results, y_test, f'{output_dir}/04_confusion_matrices_test', dataset_type='Test')

    # 5. PR曲线 - 训练集
    plot_pr_curves(results, y_train, f'{output_dir}/05_pr_curves_train', dataset_type='Train')
    # 5. PR曲线 - 测试集
    plot_pr_curves(results, y_test, f'{output_dir}/05_pr_curves_test', dataset_type='Test')

    # 6. 雷达图
    plot_radar_chart(results, f'{output_dir}/06_radar_chart')

    # 7. 特征重要性
    plot_feature_importance(results, feature_names, f'{output_dir}/07_feature_importance')

    print(f"\n所有图表已保存至: {output_dir}/")

def plot_performance_comparison(results, save_base_path):
    """性能对比柱状图（包含训练集和测试集）"""
    # 创建训练集和测试集的对比
    fig, axes = plt.subplots(2, 3, figsize=(22, 16))
    axes = axes.flatten()

    metrics_list = ['Accuracy', 'Precision', 'Recall', 'F1-score', 'AUC', 'Brier Score']

    for idx, metric in enumerate(metrics_list):
        ax = axes[idx]

        # 获取训练集和测试集的指标
        train_values = [res['metrics_train'].get(metric, 0) for res in results.values()]
        test_values = [res['metrics'].get(metric, 0) for res in results.values()]

        x = np.arange(len(results))
        width = 0.35

        bars1 = ax.bar(x - width/2, train_values, width, label='Train', color='skyblue', edgecolor='black')
        bars2 = ax.bar(x + width/2, test_values, width, label='Test', color='coral', edgecolor='black')

        ax.set_xticks(x)
        ax.set_xticklabels(results.keys(), rotation=45, ha='right', fontsize=9)
        ax.set_ylabel(metric, fontsize=11)
        ax.set_title(f'{metric} Comparison (Train vs Test)', fontsize=13, weight='bold')
        ax.legend(loc='best', fontsize=10)
        ax.grid(axis='y', alpha=0.3)

        # 添加数值标签
        for bar in bars1:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=7)
        for bar in bars2:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=7)

    plt.tight_layout()
    save_figure(fig, save_base_path)
    plt.close()

def plot_roc_curves(results, y_true, save_base_path, dataset_type='Test'):
    """ROC曲线（支持训练集和测试集）"""
    fig, ax = plt.subplots(figsize=(14, 12))

    colors = plt.cm.tab10(np.linspace(0, 1, len(results)))

    for idx, (name, res) in enumerate(results.items()):
        # 根据数据集类型选择相应的预测结果
        if dataset_type == 'Train':
            y_prob = res.get('y_prob_train', res['y_prob'])
            auc = res.get('metrics_train', res['metrics'])['AUC']
            auc_ci = res.get('metrics_train', res['metrics']).get('AUC_CI', (0, 0))
        else:
            y_prob = res['y_prob']
            auc = res['metrics']['AUC']
            auc_ci = res['metrics'].get('AUC_CI', (0, 0))

        fpr, tpr, _ = roc_curve(y_true, y_prob)
        ax.plot(fpr, tpr, color=colors[idx], lw=2.5,
                label=f'{name} (AUC={auc:.4f}, 95%CI: {auc_ci[0]:.3f}-{auc_ci[1]:.3f})')

    ax.plot([0, 1], [0, 1], 'k--', lw=2, label='Random (AUC=0.5000)')
    ax.set_xlim([-0.02, 1.0])
    ax.set_ylim([0.0, 1.02])
    ax.set_xlabel('False Positive Rate', fontsize=13)
    ax.set_ylabel('True Positive Rate', fontsize=13)
    ax.set_title(f'ROC Curves Comparison - {dataset_type} Set', fontsize=15, weight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    save_figure(fig, save_base_path)
    plt.close()

def plot_calibration_curves(results, y_true, save_base_path, dataset_type='Test'):
    """校准曲线（支持训练集和测试集）"""
    fig, ax = plt.subplots(figsize=(14, 12))

    colors = plt.cm.tab10(np.linspace(0, 1, len(results)))

    for idx, (name, res) in enumerate(results.items()):
        # 根据数据集类型选择相应的预测结果
        if dataset_type == 'Train':
            y_prob = res.get('y_prob_train', res['y_prob'])
            bs = res.get('metrics_train', res['metrics'])['Brier Score']
        else:
            y_prob = res['y_prob']
            bs = res['metrics']['Brier Score']

        prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy='uniform')
        ax.plot(prob_pred, prob_true, 's-', color=colors[idx], lw=2.5, markersize=7,
                label=f'{name} (BS={bs:.4f})')

    ax.plot([0, 1], [0, 1], 'k--', lw=2.5, label='Perfectly Calibrated')
    ax.set_xlim([-0.02, 1.0])
    ax.set_ylim([-0.02, 1.0])
    ax.set_xlabel('Mean Predicted Probability', fontsize=13)
    ax.set_ylabel('Fraction of Positives', fontsize=13)
    ax.set_title(f'Calibration Curves - {dataset_type} Set', fontsize=15, weight='bold')
    ax.legend(loc='lower right', fontsize=10)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    save_figure(fig, save_base_path)
    plt.close()

def plot_confusion_matrices(results, y_true, save_base_path, dataset_type='Test'):
    """混淆矩阵（支持训练集和测试集）"""
    n_models = len(results)
    n_cols = 3
    n_rows = (n_models + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6*n_rows))
    axes = axes.flatten() if n_models > 1 else [axes]

    for idx, (name, res) in enumerate(results.items()):
        # 根据数据集类型选择相应的预测结果
        if dataset_type == 'Train':
            y_pred = res.get('y_pred_train', res['y_pred'])
        else:
            y_pred = res['y_pred']

        cm = confusion_matrix(y_true, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                   xticklabels=['No Recurrence', 'Recurrence'],
                   yticklabels=['No Recurrence', 'Recurrence'],
                   cbar=False, annot_kws={'size': 12})
        axes[idx].set_title(f'{name} ({dataset_type})', fontsize=12, weight='bold')
        axes[idx].set_xlabel('Predicted', fontsize=10)
        axes[idx].set_ylabel('Actual', fontsize=10)

    for idx in range(n_models, len(axes)):
        axes[idx].axis('off')

    plt.tight_layout()
    save_figure(fig, save_base_path)
    plt.close()

def plot_pr_curves(results, y_true, save_base_path, dataset_type='Test'):
    """PR曲线（支持训练集和测试集，并标注AP面积）"""
    from sklearn.metrics import auc

    fig, ax = plt.subplots(figsize=(14, 12))

    colors = plt.cm.tab10(np.linspace(0, 1, len(results)))

    for idx, (name, res) in enumerate(results.items()):
        # 根据数据集类型选择相应的预测结果
        if dataset_type == 'Train':
            y_prob = res.get('y_prob_train', res['y_prob'])
            ap = res.get('metrics_train', res['metrics']).get('AP', 0)
        else:
            y_prob = res['y_prob']
            ap = res['metrics'].get('AP', 0)

        precision, recall, _ = precision_recall_curve(y_true, y_prob)
        pr_auc = auc(recall, precision)  # PR曲线下面积

        # 使用AP或PR_AUC
        display_ap = ap if ap > 0 else pr_auc

        ax.plot(recall, precision, color=colors[idx], lw=2.5,
                label=f'{name} (AP={display_ap:.4f})')

        # 添加面积填充显示
        ax.fill_between(recall, precision, alpha=0.05, color=colors[idx])

    baseline = np.mean(y_true)
    ax.axhline(y=baseline, color='k', linestyle='--', lw=2, label=f'Baseline ({baseline:.3f})')
    ax.set_xlabel('Recall', fontsize=13)
    ax.set_ylabel('Precision', fontsize=13)
    ax.set_title(f'Precision-Recall Curves - {dataset_type} Set\n(AP = Average Precision)', fontsize=15, weight='bold')
    ax.legend(loc='lower left', fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.set_xlim([0.0, 1.05])
    ax.set_ylim([0.0, 1.05])

    plt.tight_layout()
    save_figure(fig, save_base_path)
    plt.close()

def plot_radar_chart(results, save_base_path):
    """雷达图"""
    from math import pi

    metrics = ['Accuracy', 'Precision', 'Recall', 'F1-score', 'AUC']
    sorted_models = sorted(results.items(),
                          key=lambda x: x[1]['metrics']['AUC'],
                          reverse=True)[:4]

    fig, axes = plt.subplots(2, 2, figsize=(16, 16), subplot_kw=dict(polar=True))
    axes = axes.flatten()

    angles = [n / float(len(metrics)) * 2 * pi for n in range(len(metrics))]
    angles += angles[:1]

    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

    for idx, ((name, res), color) in enumerate(zip(sorted_models, colors)):
        ax = axes[idx]

        values = [res['metrics'][m] for m in metrics]
        values += values[:1]

        ax.plot(angles, values, 'o-', linewidth=3, color=color, markersize=8)
        ax.fill(angles, values, alpha=0.25, color=color)
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(metrics, fontsize=11)
        ax.set_ylim(0, 1)
        ax.set_title(name, size=14, weight='bold', pad=20)
        ax.grid(True)

        for angle, value in zip(angles, values):
            ax.text(angle, value+0.08, f'{value:.2f}', ha='center', va='center', fontsize=9)

    plt.tight_layout()
    save_figure(fig, save_base_path)
    plt.close()

def plot_feature_importance(results, feature_names, save_base_path):
    """特征重要性"""
    tree_models = ['Decision Tree', 'Random Forest', 'Gradient Boosting',
                   'XGBoost', 'LightGBM', 'CatBoost', 'Stacking (Paper Method)', 'Stacking v2 (Top3)']

    available_models = [name for name in tree_models if name in results]

    if not available_models:
        print("无树模型可用，跳过特征重要性图")
        return

    n_models = len(available_models)
    fig, axes = plt.subplots((n_models+1)//2, 2, figsize=(18, 5*n_models))
    if n_models == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for idx, name in enumerate(available_models):
        model = results[name]['model']

        if hasattr(model, 'feature_importances_'):
            importances = model.feature_importances_
            plot_names = feature_names
        elif name in ['Stacking (Paper Method)', 'Stacking v2 (Top3)']:
            meta = model.final_estimator_
            if hasattr(meta, 'feature_importances_'):
                importances = meta.feature_importances_
                plot_names = [n for n, _ in model.estimators]
            else:
                continue
        else:
            continue

        if len(importances) == len(plot_names):
            indices = np.argsort(importances)[::-1][:10]

            axes[idx].barh(range(len(indices)), importances[indices], color='steelblue')
            axes[idx].set_yticks(range(len(indices)))
            axes[idx].set_yticklabels([plot_names[i] for i in indices], fontsize=10)
            axes[idx].set_xlabel('Importance', fontsize=11)
            axes[idx].set_title(f'{name} - Top 10 Features', fontsize=12, weight='bold')
            axes[idx].invert_yaxis()

    for idx in range(n_models, len(axes)):
        axes[idx].axis('off')

    plt.tight_layout()
    save_figure(fig, save_base_path)
    plt.close()

# ============================================================
# 第八部分: 结果保存
# ============================================================

def save_results(results, X_train, X_test, y_train, y_test, output_dir='./results'):
    """
    保存所有结果，包括训练集和测试集的预测结果
    """
    print("\n" + "=" * 70)
    print("步骤 7: 保存结果")
    print("=" * 70)

    os.makedirs(output_dir, exist_ok=True)

    # 测试集性能表
    metrics_df = pd.DataFrame({name: res['metrics'] for name, res in results.items()}).T
    metrics_df = metrics_df.round(4)
    metrics_df.to_csv(f'{output_dir}/model_performance_test.csv')
    metrics_df.to_excel(f'{output_dir}/model_performance_test.xlsx')

    # 训练集性能表
    metrics_train_df = pd.DataFrame({name: res['metrics_train'] for name, res in results.items()}).T
    metrics_train_df = metrics_train_df.round(4)
    metrics_train_df.to_csv(f'{output_dir}/model_performance_train.csv')
    metrics_train_df.to_excel(f'{output_dir}/model_performance_train.xlsx')

    # 合并性能表
    combined_metrics = {}
    for name in results.keys():
        test_metrics = {f"Test_{k}": v for k, v in results[name]['metrics'].items()}
        train_metrics = {f"Train_{k}": v for k, v in results[name]['metrics_train'].items()}
        combined_metrics[name] = {**test_metrics, **train_metrics}
    combined_df = pd.DataFrame(combined_metrics).T
    combined_df = combined_df.round(4)
    combined_df.to_csv(f'{output_dir}/model_performance_combined.csv')
    combined_df.to_excel(f'{output_dir}/model_performance_combined.xlsx')

    # 排名
    print("\n模型排名 (按测试集AUC):")
    ranking = metrics_df.sort_values('AUC', ascending=False)
    for idx, (name, row) in enumerate(ranking.iterrows(), 1):
        marker = " ⭐" if idx == 1 else ""
        auc_ci = row.get('AUC_CI', (0, 0))
        print(f"{idx:2d}. {name:30s} AUC: {row['AUC']:.4f} (95%CI: {auc_ci[0]:.3f}-{auc_ci[1]:.3f})  F1: {row['F1-score']:.4f}{marker}")

    # 保存最佳模型
    best_name = ranking.index[0]
    joblib.dump(results[best_name]['model'], f'{output_dir}/BEST_MODEL_{best_name.replace(" ", "_")}.pkl')

    # 保存所有模型
    for name, res in results.items():
        joblib.dump(res['model'], f'{output_dir}/model_{name.replace(" ", "_")}.pkl')

    # 保存测试集预测结果
    pred_test_df = pd.DataFrame({
        'True_Label': y_test.values if hasattr(y_test, 'values') else y_test,
        **{f'{name}_Prob': res['y_prob'] for name, res in results.items()},
        **{f'{name}_Pred': res['y_pred'] for name, res in results.items()}
    })
    pred_test_df.to_csv(f'{output_dir}/test_predictions.csv', index=False)
    print(f"\n✓ 保存测试集预测结果: test_predictions.csv")

    # 保存训练集预测结果
    pred_train_df = pd.DataFrame({
        'True_Label': y_train.values if hasattr(y_train, 'values') else y_train,
        **{f'{name}_Prob_Train': res['y_prob_train'] for name, res in results.items()},
        **{f'{name}_Pred_Train': res['y_pred_train'] for name, res in results.items()}
    })
    pred_train_df.to_csv(f'{output_dir}/train_predictions.csv', index=False)
    print(f"✓ 保存训练集预测结果: train_predictions.csv")

    # 保存所有预测结果（合并）
    # 由于训练集和测试集行数不同，分别保存
    all_predictions = {
        'train': pred_train_df,
        'test': pred_test_df
    }

    # 保存排名
    ranking.to_csv(f'{output_dir}/model_ranking.csv')

    print(f"\n所有结果保存在: {output_dir}/")
    return metrics_df, metrics_train_df, ranking

# ============================================================
# 第九部分: SHAP分析（支持PDF）
# ============================================================


# ============================================================
# 新增: SHAP特征重要性保存函数
# ============================================================

def save_shap_feature_importance(shap_values, X_data, feature_names, model_name, output_dir='./results'):
    """
    保存SHAP特征重要性结果到CSV和XLSX文件
    """
    import os
    
    print(f"\n{'='*70}")
    print(f"保存SHAP特征重要性结果: {model_name}")
    print(f"{'='*70}")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. 计算全局特征重要性 (Mean Absolute SHAP values)
    mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
    std_abs_shap = np.std(np.abs(shap_values), axis=0)
    
    # 创建特征重要性DataFrame
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'Mean_Abs_SHAP': mean_abs_shap,
        'Std_Abs_SHAP': std_abs_shap,
        'Mean_SHAP': np.mean(shap_values, axis=0),
        'Std_SHAP': np.std(shap_values, axis=0),
        'Max_Abs_SHAP': np.max(np.abs(shap_values), axis=0),
        'Min_SHAP': np.min(shap_values, axis=0),
        'Max_SHAP': np.max(shap_values, axis=0)
    })
    
    # 按重要性排序
    importance_df = importance_df.sort_values('Mean_Abs_SHAP', ascending=False)
    importance_df['Rank'] = range(1, len(importance_df) + 1)
    
    # 重新排列列顺序
    importance_df = importance_df[['Rank', 'Feature', 'Mean_Abs_SHAP', 'Std_Abs_SHAP', 
                                     'Mean_SHAP', 'Std_SHAP', 'Max_Abs_SHAP', 
                                     'Min_SHAP', 'Max_SHAP']]
    
    # 保存特征重要性
    importance_df.to_csv(f'{output_dir}/shap_feature_importance_{model_name.replace(" ", "_")}.csv', index=False)
    importance_df.to_excel(f'{output_dir}/shap_feature_importance_{model_name.replace(" ", "_")}.xlsx', index=False)
    
    print(f"\n全局特征重要性排名 (Top 10):")
    print(importance_df[['Rank', 'Feature', 'Mean_Abs_SHAP']].head(10).to_string(index=False))
    
    # 2. 保存详细的SHAP值矩阵
    shap_values_df = pd.DataFrame(shap_values, columns=feature_names)
    shap_values_df.to_csv(f'{output_dir}/shap_values_matrix_{model_name.replace(" ", "_")}.csv', index=False)
    
    # 3. 保存特征统计摘要
    summary_stats = {
        'Metric': [
            'Total_Samples',
            'Num_Features',
            'Model_Name',
            'Top_Feature',
            'Top_Feature_Importance',
            'Top_3_Cumulative_Importance',
            'Top_5_Cumulative_Importance'
        ],
        'Value': [
            len(X_data),
            len(feature_names),
            model_name,
            importance_df.iloc[0]['Feature'],
            f"{importance_df.iloc[0]['Mean_Abs_SHAP']:.6f}",
            f"{importance_df.head(3)['Mean_Abs_SHAP'].sum():.6f}",
            f"{importance_df.head(5)['Mean_Abs_SHAP'].sum():.6f}"
        ]
    }
    summary_df = pd.DataFrame(summary_stats)
    summary_df.to_csv(f'{output_dir}/shap_summary_{model_name.replace(" ", "_")}.csv', index=False)
    summary_df.to_excel(f'{output_dir}/shap_summary_{model_name.replace(" ", "_")}.xlsx', index=False)
    
    # 4. 创建综合SHAP报告（多sheet Excel）
    try:
        with pd.ExcelWriter(f'{output_dir}/shap_complete_report_{model_name.replace(" ", "_")}.xlsx', engine='openpyxl') as writer:
            # Sheet 1: 特征重要性排名
            importance_df.to_excel(writer, sheet_name='Feature_Importance_Ranking', index=False)
            
            # Sheet 2: 统计摘要
            summary_df.to_excel(writer, sheet_name='Summary_Statistics', index=False)
            
            # Sheet 3: SHAP值矩阵（前100个样本）
            shap_values_df.head(100).to_excel(writer, sheet_name='SHAP_Values_Sample', index=False)
            
            # Sheet 4: 特征间SHAP相关性
            if len(feature_names) > 1:
                shap_corr = np.corrcoef(shap_values.T)
                shap_corr_df = pd.DataFrame(shap_corr, index=feature_names, columns=feature_names)
                shap_corr_df.to_excel(writer, sheet_name='Feature_SHAP_Correlation')
        
        print(f"\n✓ SHAP结果已保存:")
        print(f"  - {output_dir}/shap_feature_importance_{model_name.replace(' ', '_')}.csv/xlsx")
        print(f"  - {output_dir}/shap_values_matrix_{model_name.replace(' ', '_')}.csv")
        print(f"  - {output_dir}/shap_summary_{model_name.replace(' ', '_')}.csv/xlsx")
        print(f"  - {output_dir}/shap_complete_report_{model_name.replace(' ', '_')}.xlsx (综合报告)")
    except Exception as e:
        print(f"警告: 保存综合报告时出错: {e}")
        print(f"但单独的文件已保存成功")
    
    return importance_df

def shap_analysis(results, X_train, X_test, feature_names, output_dir='./results'):
    """
    SHAP可解释性分析
    """
    try:
        import shap
        print("\n" + "=" * 70)
        print("步骤 8: SHAP可解释性分析")
        print("=" * 70)

        candidates = [
            # 'XGBoost', 
            'LightGBM'
            # ,
            # 'CatBoost', 'Random Forest', 'Gradient Boosting'
        ]
        target = next((name for name in candidates if name in results), None)

        if not target:
            print("无合适模型进行SHAP分析")
            return

        print(f"分析模型: {target}")
        model = results[target]['model']

        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_test)

        if isinstance(shap_values, list):
            shap_values = shap_values[1]

        # 全局解释 - 条形图
        fig = plt.figure(figsize=(12, 10))
        shap.summary_plot(shap_values, X_test, feature_names=feature_names,
                         plot_type="bar", show=False)
        plt.title(f'{target} - Global Feature Importance', fontsize=14, weight='bold')
        plt.tight_layout()
        save_figure(fig, f'{output_dir}/08_shap_global')
        plt.close()

        # 蜂群图
        fig = plt.figure(figsize=(12, 10))
        shap.summary_plot(shap_values, X_test, feature_names=feature_names, show=False)
        plt.title(f'{target} - SHAP Beeswarm', fontsize=14, weight='bold')
        plt.tight_layout()
        save_figure(fig, f'{output_dir}/09_shap_beeswarm')
        plt.close()

        # 局部解释
        y_prob = results[target]['y_prob']
        high_idx = np.argmax(y_prob)
        low_idx = np.argmin(y_prob)

        # 高风险
        fig = plt.figure(figsize=(14, 10))
        shap.waterfall_plot(shap.Explanation(
            values=shap_values[high_idx],
            base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
            data=X_test.iloc[high_idx],
            feature_names=feature_names
        ), max_display=15, show=False)
        plt.title(f'High Risk Case (Prob={y_prob[high_idx]:.3f})', fontsize=14, weight='bold')
        plt.tight_layout()
        save_figure(fig, f'{output_dir}/10_shap_high_risk')
        plt.close()

        # 低风险
        fig = plt.figure(figsize=(14, 10))
        shap.waterfall_plot(shap.Explanation(
            values=shap_values[low_idx],
            base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, list) else explainer.expected_value,
            data=X_test.iloc[low_idx],
            feature_names=feature_names
        ), max_display=15, show=False)
        plt.title(f'Low Risk Case (Prob={y_prob[low_idx]:.3f})', fontsize=14, weight='bold')
        plt.tight_layout()
        save_figure(fig, f'{output_dir}/11_shap_low_risk')
        plt.close()

        # 保存SHAP特征重要性数据到文件
        save_shap_feature_importance(shap_values, X_test, feature_names, target, output_dir)

        print("SHAP分析完成")

    except ImportError:
        print("SHAP未安装，跳过")

甲状腺癌复发预测 - 10模型对比分析系统（PDF输出版）


In [2]:
# ============================================================
# 新增部分: Cohen Kappa 系数计算与可视化
# ============================================================

def calculate_cohen_kappa_matrix(results, y_true, dataset_type='Test'):
    """
    计算模型之间的Cohen Kappa系数矩阵
    """
    print(f"\n{'='*70}")
    print(f"计算Cohen Kappa系数矩阵 - {dataset_type} Set")
    print(f"{'='*70}")
    
    model_names = list(results.keys())
    n_models = len(model_names)
    
    # 初始化Kappa矩阵
    kappa_matrix = np.zeros((n_models, n_models))
    
    # 获取预测结果
    predictions = {}
    for name, res in results.items():
        if dataset_type == 'Train':
            predictions[name] = res.get('y_pred_train', res['y_pred'])
        else:
            predictions[name] = res['y_pred']
    
    # 计算Kappa系数
    for i, model_i in enumerate(model_names):
        for j, model_j in enumerate(model_names):
            if i == j:
                kappa_matrix[i, j] = 1.0  # 对角线为1
            elif i < j:
                # 计算两个模型预测之间的Kappa
                kappa = cohen_kappa_score(
                    predictions[model_i], 
                    predictions[model_j]
                )
                kappa_matrix[i, j] = kappa
                kappa_matrix[j, i] = kappa  # 对称矩阵
    
    # 创建DataFrame
    kappa_df = pd.DataFrame(kappa_matrix, index=model_names, columns=model_names)
    
    # 打印结果
    print(f"\nCohen Kappa系数矩阵:")
    print(kappa_df.round(4))
    
    # 计算统计摘要
    off_diagonal = kappa_matrix[np.triu_indices_from(kappa_matrix, k=1)]
    print(f"\nKappa系数统计摘要 (非对角线元素):")
    print(f"  平均值: {np.mean(off_diagonal):.4f}")
    print(f"  标准差: {np.std(off_diagonal):.4f}")
    print(f"  最小值: {np.min(off_diagonal):.4f}")
    print(f"  最大值: {np.max(off_diagonal):.4f}")
    
    return kappa_df

def plot_kappa_heatmap(kappa_df, save_base_path, dataset_type='Test'):
    """
    绘制Cohen Kappa系数热图
    """
    fig, ax = plt.subplots(figsize=(16, 14))
    
    # 创建掩码，隐藏下三角（保留对角线）
    mask = np.tril(np.ones_like(kappa_df.values, dtype=bool), k=-1)
    
    # 绘制热图
    sns.heatmap(
        kappa_df, 
        annot=True, 
        fmt='.3f', 
        cmap='RdYlGn',  # 红-黄-绿配色
        center=0.5,
        vmin=0, 
        vmax=1,
        square=True,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8, "label": "Cohen's Kappa"},
        annot_kws={'size': 10, 'weight': 'bold'},
        mask=mask,
        ax=ax
    )
    
    # 设置标题和标签
    ax.set_title(f"Model Agreement - Cohen's Kappa Heatmap ({dataset_type} Set)\n" + 
                 "(Higher values indicate better agreement between models)", 
                 fontsize=14, weight='bold', pad=20)
    
    # 旋转x轴标签
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(rotation=0, fontsize=10)
    
    plt.tight_layout()
    save_figure(fig, save_base_path)
    plt.close()
    print(f"  ✓ Kappa热图已保存: {save_base_path}")

def save_kappa_results(kappa_df_train, kappa_df_test, output_dir='./results'):
    """
    保存Kappa系数结果到CSV和Excel
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # 保存训练集结果
    kappa_df_train.to_csv(f'{output_dir}/cohen_kappa_matrix_train.csv')
    kappa_df_train.to_excel(f'{output_dir}/cohen_kappa_matrix_train.xlsx')
    
    # 保存测试集结果
    kappa_df_test.to_csv(f'{output_dir}/cohen_kappa_matrix_test.csv')
    kappa_df_test.to_excel(f'{output_dir}/cohen_kappa_matrix_test.xlsx')
    
    print(f"\n✓ Kappa系数矩阵已保存:")
    print(f"  - {output_dir}/cohen_kappa_matrix_train.csv/xlsx")
    print(f"  - {output_dir}/cohen_kappa_matrix_test.csv/xlsx")


# ============================================================
# 新增部分: Delong检验实现
# ============================================================

def delong_test(y_true, y_score1, y_score2):
    """
    实现Delong检验来比较两个模型的AUC差异
    
    参考: Delong et al., Biometrics 1988
    """
    y_true = np.array(y_true)
    y_score1 = np.array(y_score1)
    y_score2 = np.array(y_score2)
    
    # 获取正负样本索引
    pos_idx = np.where(y_true == 1)[0]
    neg_idx = np.where(y_true == 0)[0]
    
    n_pos = len(pos_idx)
    n_neg = len(neg_idx)
    
    if n_pos == 0 or n_neg == 0:
        return 0, 1.0, 0.5, 0.5
    
    # 计算AUC
    auc1 = roc_auc_score(y_true, y_score1)
    auc2 = roc_auc_score(y_true, y_score2)
    
    # 计算V10和V01统计量
    def compute_v_stats(y_score, pos_idx, neg_idx):
        n_pos = len(pos_idx)
        n_neg = len(neg_idx)
        
        v10 = np.zeros(n_pos)
        for i, idx in enumerate(pos_idx):
            v10[i] = np.sum(y_score[neg_idx] < y_score[idx]) + 0.5 * np.sum(y_score[neg_idx] == y_score[idx])
        v10 = v10 / n_neg
        
        v01 = np.zeros(n_neg)
        for j, idx in enumerate(neg_idx):
            v01[j] = np.sum(y_score[pos_idx] > y_score[idx]) + 0.5 * np.sum(y_score[pos_idx] == y_score[idx])
        v01 = v01 / n_pos
        
        return v10, v01
    
    v10_1, v01_1 = compute_v_stats(y_score1, pos_idx, neg_idx)
    v10_2, v01_2 = compute_v_stats(y_score2, pos_idx, neg_idx)
    
    # 计算协方差和方差
    s10 = np.cov(v10_1, v10_2)[0, 1] if n_pos > 1 else 0
    s01 = np.cov(v01_1, v01_2)[0, 1] if n_neg > 1 else 0
    
    var1 = (np.var(v10_1) / n_pos + np.var(v01_1) / n_neg) if n_pos > 1 and n_neg > 1 else 0.01
    var2 = (np.var(v10_2) / n_pos + np.var(v01_2) / n_neg) if n_pos > 1 and n_neg > 1 else 0.01
    
    cov = s10 / n_pos + s01 / n_neg
    
    # 计算Z统计量
    var_diff = var1 + var2 - 2 * cov
    if var_diff <= 0:
        var_diff = 0.001
    
    z_statistic = (auc1 - auc2) / np.sqrt(var_diff)
    
    # 计算双尾p值
    p_value = 2 * (1 - stats.norm.cdf(abs(z_statistic)))
    
    return z_statistic, p_value, auc1, auc2

def calculate_delong_matrix(results, y_true, dataset_type='Test'):
    """
    计算模型之间AUC比较的Delong检验矩阵
    """
    print(f"\n{'='*70}")
    print(f"计算Delong检验矩阵 - {dataset_type} Set")
    print(f"{'='*70}")
    
    model_names = list(results.keys())
    n_models = len(model_names)
    
    # 初始化矩阵
    pvalue_matrix = np.ones((n_models, n_models))
    zstat_matrix = np.zeros((n_models, n_models))
    auc_diff_matrix = np.zeros((n_models, n_models))
    
    # 获取预测概率
    probabilities = {}
    auc_values = {}
    for name, res in results.items():
        if dataset_type == 'Train':
            y_prob = res.get('y_prob_train', res['y_prob'])
            metrics = res.get('metrics_train', res['metrics'])
        else:
            y_prob = res['y_prob']
            metrics = res['metrics']
        
        probabilities[name] = y_prob
        auc_values[name] = metrics['AUC']
    
    # 计算Delong检验
    print(f"\n正在执行Delong检验 (共 {n_models*(n_models-1)//2} 对比较)...")
    
    for i, model_i in enumerate(model_names):
        for j, model_j in enumerate(model_names):
            if i == j:
                pvalue_matrix[i, j] = 1.0
                zstat_matrix[i, j] = 0.0
                auc_diff_matrix[i, j] = 0.0
            elif i < j:
                z_stat, p_val, auc1, auc2 = delong_test(
                    y_true,
                    probabilities[model_i],
                    probabilities[model_j]
                )
                
                pvalue_matrix[i, j] = p_val
                pvalue_matrix[j, i] = p_val
                zstat_matrix[i, j] = z_stat
                zstat_matrix[j, i] = -z_stat
                auc_diff = auc1 - auc2
                auc_diff_matrix[i, j] = auc_diff
                auc_diff_matrix[j, i] = -auc_diff
    
    # 创建DataFrames
    pvalue_df = pd.DataFrame(pvalue_matrix, index=model_names, columns=model_names)
    zstat_df = pd.DataFrame(zstat_matrix, index=model_names, columns=model_names)
    auc_diff_df = pd.DataFrame(auc_diff_matrix, index=model_names, columns=model_names)
    
    # 打印显著性结果
    print(f"\n显著性结果 (p < 0.05):")
    significant_pairs = []
    for i in range(n_models):
        for j in range(i+1, n_models):
            p_val = pvalue_matrix[i, j]
            auc1 = auc_values[model_names[i]]
            auc2 = auc_values[model_names[j]]
            if p_val < 0.05:
                better = model_names[i] if auc1 > auc2 else model_names[j]
                significant_pairs.append((model_names[i], model_names[j], p_val, better))
                print(f"  {model_names[i]} vs {model_names[j]}: p={p_val:.4f} (更好: {better})")
    
    if not significant_pairs:
        print("  未发现显著差异 (所有p值 >= 0.05)")
    
    # 统计信息
    off_diagonal_p = pvalue_matrix[np.triu_indices_from(pvalue_matrix, k=1)]
    print(f"\nDelong检验p值统计摘要:")
    print(f"  p < 0.05 的比例: {np.mean(off_diagonal_p < 0.05):.1%}")
    print(f"  p值范围: [{np.min(off_diagonal_p):.4f}, {np.max(off_diagonal_p):.4f}]")
    
    return pvalue_df, zstat_df, auc_diff_df

def plot_delong_heatmap(pvalue_df, zstat_df, auc_diff_df, save_base_path, dataset_type='Test'):
    """
    绘制Delong检验结果热图
    """
    fig, axes = plt.subplots(1, 3, figsize=(24, 10))
    
    # 创建掩码
    mask = np.tril(np.ones_like(pvalue_df.values, dtype=bool), k=-1)
    
    # 1. p值热图
    ax1 = axes[0]
    sns.heatmap(
        pvalue_df, 
        annot=True, 
        fmt='.3f',
        cmap='RdYlGn_r',
        vmin=0, 
        vmax=0.1,
        square=True,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8, "label": "p-value"},
        annot_kws={'size': 9},
        mask=mask,
        ax=ax1
    )
    ax1.set_title(f'Delong Test p-values ({dataset_type})\n(Green: Significant p<0.05)', 
                  fontsize=12, weight='bold')
    ax1.set_xticklabels(pvalue_df.columns, rotation=45, ha='right', fontsize=9)
    ax1.set_yticklabels(pvalue_df.index, rotation=0, fontsize=9)
    
    # 2. Z统计量热图
    ax2 = axes[1]
    z_max = np.max(np.abs(zstat_df.values))
    sns.heatmap(
        zstat_df, 
        annot=True, 
        fmt='.2f',
        cmap='RdBu_r',
        center=0,
        vmin=-z_max, 
        vmax=z_max,
        square=True,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8, "label": "Z-statistic"},
        annot_kws={'size': 9},
        mask=mask,
        ax=ax2
    )
    ax2.set_title(f'Delong Z-statistics ({dataset_type})\n(Positive: Row model better)', 
                  fontsize=12, weight='bold')
    ax2.set_xticklabels(zstat_df.columns, rotation=45, ha='right', fontsize=9)
    ax2.set_yticklabels(zstat_df.index, rotation=0, fontsize=9)
    
    # 3. AUC差异热图
    ax3 = axes[2]
    diff_max = np.max(np.abs(auc_diff_df.values))
    sns.heatmap(
        auc_diff_df, 
        annot=True, 
        fmt='.4f',
        cmap='RdBu_r',
        center=0,
        vmin=-diff_max, 
        vmax=diff_max,
        square=True,
        linewidths=0.5,
        cbar_kws={"shrink": 0.8, "label": "AUC Difference"},
        annot_kws={'size': 9},
        mask=mask,
        ax=ax3
    )
    ax3.set_title(f'AUC Difference Matrix ({dataset_type})\n(Row - Column)', 
                  fontsize=12, weight='bold')
    ax3.set_xticklabels(auc_diff_df.columns, rotation=45, ha='right', fontsize=9)
    ax3.set_yticklabels(auc_diff_df.index, rotation=0, fontsize=9)
    
    plt.suptitle(f'Delong Test for AUC Comparison - {dataset_type} Set', 
                 fontsize=16, weight='bold', y=1.02)
    plt.tight_layout()
    save_figure(fig, save_base_path)
    plt.close()
    print(f"  ✓ Delong检验热图已保存: {save_base_path}")

def save_delong_results(pvalue_train, pvalue_test, zstat_train, zstat_test, 
                        auc_diff_train, auc_diff_test, output_dir='./results'):
    """
    保存Delong检验结果到CSV和Excel
    """
    os.makedirs(output_dir, exist_ok=True)
    
    # 保存训练集结果
    pvalue_train.to_csv(f'{output_dir}/delong_pvalues_train.csv')
    pvalue_train.to_excel(f'{output_dir}/delong_pvalues_train.xlsx')
    zstat_train.to_csv(f'{output_dir}/delong_zstatistics_train.csv')
    auc_diff_train.to_csv(f'{output_dir}/delong_auc_diff_train.csv')
    
    # 保存测试集结果
    pvalue_test.to_csv(f'{output_dir}/delong_pvalues_test.csv')
    pvalue_test.to_excel(f'{output_dir}/delong_pvalues_test.xlsx')
    zstat_test.to_csv(f'{output_dir}/delong_zstatistics_test.csv')
    auc_diff_test.to_csv(f'{output_dir}/delong_auc_diff_test.csv')
    
    print(f"\n✓ Delong检验结果已保存:")
    print(f"  - {output_dir}/delong_pvalues_train/test.csv/xlsx")
    print(f"  - {output_dir}/delong_zstatistics_train/test.csv")
    print(f"  - {output_dir}/delong_auc_diff_train/test.csv")


# ============================================================
# 新增部分: 校准曲线详细分析
# ============================================================

def calculate_calibration_metrics(y_true, y_prob, n_bins=10):
    """
    计算校准曲线的详细指标
    """
    from sklearn.calibration import calibration_curve
    
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins, strategy='uniform')
    
    # 计算校准误差 (ECE - Expected Calibration Error)
    bin_counts, bin_edges = np.histogram(y_prob, bins=n_bins, range=(0, 1))
    total_samples = len(y_true)
    
    ece = 0
    for i in range(len(prob_true)):
        if bin_counts[i] > 0:
            bin_weight = bin_counts[i] / total_samples
            ece += bin_weight * abs(prob_true[i] - prob_pred[i])
    
    # 最大校准误差 (MCE)
    mce = np.max(np.abs(prob_true - prob_pred)) if len(prob_true) > 0 else 0
    
    # Brier Score 已在其他部分计算
    brier = brier_score_loss(y_true, y_prob)
    
    metrics = {
        'ECE': ece,  # Expected Calibration Error
        'MCE': mce,  # Maximum Calibration Error
        'Brier Score': brier,
        'prob_true': prob_true,
        'prob_pred': prob_pred,
        'bin_counts': bin_counts
    }
    
    return metrics

def save_calibration_results(results, y_train, y_test, output_dir='./results'):
    """
    保存校准曲线详细数据到文件
    """
    print(f"\n{'='*70}")
    print("保存校准曲线详细分析结果")
    print(f"{'='*70}")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # 训练集校准指标
    calibration_metrics_train = {}
    calibration_metrics_test = {}
    
    for name, res in results.items():
        # 训练集
        y_prob_train = res.get('y_prob_train', res['y_prob'])
        metrics_train = calculate_calibration_metrics(y_train, y_prob_train)
        calibration_metrics_train[name] = {
            'ECE': metrics_train['ECE'],
            'MCE': metrics_train['MCE'],
            'Brier Score': metrics_train['Brier Score']
        }
        
        # 测试集
        y_prob_test = res['y_prob']
        metrics_test = calculate_calibration_metrics(y_test, y_prob_test)
        calibration_metrics_test[name] = {
            'ECE': metrics_test['ECE'],
            'MCE': metrics_test['MCE'],
            'Brier Score': metrics_test['Brier Score']
        }
    
    # 保存为DataFrame
    cal_train_df = pd.DataFrame(calibration_metrics_train).T
    cal_test_df = pd.DataFrame(calibration_metrics_test).T
    
    cal_train_df.to_csv(f'{output_dir}/calibration_metrics_train.csv')
    cal_train_df.to_excel(f'{output_dir}/calibration_metrics_train.xlsx')
    cal_test_df.to_csv(f'{output_dir}/calibration_metrics_test.csv')
    cal_test_df.to_excel(f'{output_dir}/calibration_metrics_test.xlsx')
    
    print(f"\n校准指标统计:")
    print(f"\n训练集:")
    print(cal_train_df.round(4))
    print(f"\n测试集:")
    print(cal_test_df.round(4))
    
    print(f"\n✓ 校准分析结果已保存:")
    print(f"  - {output_dir}/calibration_metrics_train.csv/xlsx")
    print(f"  - {output_dir}/calibration_metrics_test.csv/xlsx")
    
    return cal_train_df, cal_test_df


# ============================================================
# 新增部分: 决策曲线分析 (DCA)
# ============================================================

def decision_curve_analysis(y_true, y_prob, thresholds=None):
    """
    执行决策曲线分析 (Decision Curve Analysis)
    
    计算在不同概率阈值下的净收益 (Net Benefit)
    
    Parameters:
    -----------
    y_true : array-like
        真实标签
    y_prob : array-like
        预测概率
    thresholds : array-like, optional
        阈值范围，默认 0.01 到 0.99
        
    Returns:
    --------
    dict: 包含各阈值下的净收益
    """
    if thresholds is None:
        thresholds = np.arange(0.01, 0.99, 0.01)
    
    y_true = np.array(y_true)
    y_prob = np.array(y_prob)
    
    n = len(y_true)
    prevalence = np.mean(y_true)
    
    net_benefits = []
    
    for threshold in thresholds:
        # 根据阈值进行预测
        y_pred = (y_prob >= threshold).astype(int)
        
        # 计算 TP, FP
        tp = np.sum((y_pred == 1) & (y_true == 1))
        fp = np.sum((y_pred == 1) & (y_true == 0))
        
        # 计算净收益
        # Net Benefit = TP/n - FP/n * (threshold/(1-threshold))
        net_benefit = (tp / n) - (fp / n) * (threshold / (1 - threshold))
        net_benefits.append(net_benefit)
    
    # 计算Treat All的净收益（无论预测如何，全部治疗）
    net_benefit_all = np.full_like(thresholds, prevalence) - (1 - prevalence) * (thresholds / (1 - thresholds))
    
    # 计算Treat None的净收益（从不治疗）
    net_benefit_none = np.zeros_like(thresholds)
    
    return {
        'thresholds': thresholds,
        'net_benefits': np.array(net_benefits),
        'net_benefit_all': net_benefit_all,
        'net_benefit_none': net_benefit_none,
        'prevalence': prevalence
    }

def plot_decision_curves(results, y_true, save_base_path, dataset_type='Test'):
    """
    绘制决策曲线分析图
    """
    fig, ax = plt.subplots(figsize=(14, 10))
    
    colors = plt.cm.tab10(np.linspace(0, 1, len(results)))
    thresholds = np.arange(0.01, 0.99, 0.01)
    
    # 首先绘制Treat All和Treat None基线
    prevalence = np.mean(y_true)
    net_benefit_all = np.full_like(thresholds, prevalence) - (1 - prevalence) * (thresholds / (1 - thresholds))
    net_benefit_none = np.zeros_like(thresholds)
    
    # 限制Treat All的合理范围
    net_benefit_all = np.clip(net_benefit_all, -0.5, 1.0)
    
    ax.plot(thresholds, net_benefit_all, 'k--', lw=2, label='Treat All', alpha=0.7)
    ax.plot(thresholds, net_benefit_none, 'k:', lw=2, label='Treat None', alpha=0.7)
    
    # 绘制每个模型的决策曲线
    for idx, (name, res) in enumerate(results.items()):
        if dataset_type == 'Train':
            y_prob = res.get('y_prob_train', res['y_prob'])
        else:
            y_prob = res['y_prob']
        
        dca = decision_curve_analysis(y_true, y_prob, thresholds)
        
        # 计算AUC作为性能参考
        auc = res['metrics']['AUC'] if dataset_type == 'Test' else res.get('metrics_train', res['metrics'])['AUC']
        
        ax.plot(dca['thresholds'], dca['net_benefits'], 
                color=colors[idx], lw=2.5, label=f'{name} (AUC={auc:.3f})')
    
    ax.set_xlim([0, 1])
    ax.set_ylim([-0.1, max(0.6, np.max(net_benefit_all) * 1.1)])
    ax.set_xlabel('Threshold Probability', fontsize=13)
    ax.set_ylabel('Net Benefit', fontsize=13)
    ax.set_title(f'Decision Curve Analysis - {dataset_type} Set\n' +
                 '(Higher net benefit indicates better clinical utility)', 
                 fontsize=15, weight='bold')
    ax.legend(loc='upper right', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # 添加临床意义说明
    ax.axhline(y=0, color='gray', linestyle='-', lw=0.5, alpha=0.5)
    
    plt.tight_layout()
    save_figure(fig, save_base_path)
    plt.close()
    print(f"  ✓ 决策曲线图已保存: {save_base_path}")

def calculate_dca_metrics(results, y_true, dataset_type='Test'):
    """
    计算DCA的关键指标
    """
    dca_metrics = {}
    thresholds = np.arange(0.05, 0.95, 0.05)  # 评估常用阈值范围
    
    for name, res in results.items():
        if dataset_type == 'Train':
            y_prob = res.get('y_prob_train', res['y_prob'])
        else:
            y_prob = res['y_prob']
        
        dca = decision_curve_analysis(y_true, y_prob, thresholds)
        
        # 计算关键指标
        # 1. 最大净收益
        max_net_benefit = np.max(dca['net_benefits'])
        max_nb_threshold = dca['thresholds'][np.argmax(dca['net_benefits'])]
        
        # 2. 平均净收益（在0.1-0.5范围内）
        mask = (dca['thresholds'] >= 0.1) & (dca['thresholds'] <= 0.5)
        avg_net_benefit = np.mean(dca['net_benefits'][mask])
        
        # 3. 优于Treat All的阈值范围比例
        better_than_all = np.sum(dca['net_benefits'] > dca['net_benefit_all']) / len(dca['thresholds'])
        
        # 4. 优于Treat None的阈值范围比例
        better_than_none = np.sum(dca['net_benefits'] > dca['net_benefit_none']) / len(dca['thresholds'])
        
        dca_metrics[name] = {
            'Max_Net_Benefit': max_net_benefit,
            'Max_NB_Threshold': max_nb_threshold,
            'Avg_Net_Benefit_10_50': avg_net_benefit,
            'Better_than_Treat_All': better_than_all,
            'Better_than_Treat_None': better_than_none
        }
    
    return pd.DataFrame(dca_metrics).T

def save_dca_results(results, y_train, y_test, output_dir='./results'):
    """
    保存DCA分析结果
    """
    print(f"\n{'='*70}")
    print("保存决策曲线分析 (DCA) 结果")
    print(f"{'='*70}")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # 绘制决策曲线图
    plot_decision_curves(results, y_train, f'{output_dir}/10_decision_curve_train', dataset_type='Train')
    plot_decision_curves(results, y_test, f'{output_dir}/10_decision_curve_test', dataset_type='Test')
    
    # 计算并保存DCA指标
    dca_metrics_train = calculate_dca_metrics(results, y_train, dataset_type='Train')
    dca_metrics_test = calculate_dca_metrics(results, y_test, dataset_type='Test')
    
    dca_metrics_train.to_csv(f'{output_dir}/dca_metrics_train.csv')
    dca_metrics_train.to_excel(f'{output_dir}/dca_metrics_train.xlsx')
    dca_metrics_test.to_csv(f'{output_dir}/dca_metrics_test.csv')
    dca_metrics_test.to_excel(f'{output_dir}/dca_metrics_test.xlsx')
    
    print(f"\nDCA指标统计 - 训练集:")
    print(dca_metrics_train.round(4))
    print(f"\nDCA指标统计 - 测试集:")
    print(dca_metrics_test.round(4))
    
    print(f"\n✓ DCA结果已保存:")
    print(f"  - {output_dir}/10_decision_curve_train.png/pdf")
    print(f"  - {output_dir}/10_decision_curve_test.png/pdf")
    print(f"  - {output_dir}/dca_metrics_train.csv/xlsx")
    print(f"  - {output_dir}/dca_metrics_test.csv/xlsx")
    
    return dca_metrics_train, dca_metrics_test


# ============================================================
# 新增部分: 综合分析函数
# ============================================================

def create_model_comparison_analysis(results, y_train, y_test, output_dir='./results'):
    """
    执行模型间的综合分析
    包括: Cohen Kappa、Delong检验、校准曲线分析、决策曲线分析(DCA)
    """
    print("\n" + "=" * 70)
    print("新增步骤: 模型间比较分析 (Cohen Kappa + Delong Test)")
    print("=" * 70)
    
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Cohen Kappa 分析 - 训练集
    kappa_train = calculate_cohen_kappa_matrix(results, y_train, dataset_type='Train')
    plot_kappa_heatmap(kappa_train, f'{output_dir}/08_cohen_kappa_heatmap_train', dataset_type='Train')
    
    # 2. Cohen Kappa 分析 - 测试集
    kappa_test = calculate_cohen_kappa_matrix(results, y_test, dataset_type='Test')
    plot_kappa_heatmap(kappa_test, f'{output_dir}/08_cohen_kappa_heatmap_test', dataset_type='Test')
    
    # 保存Kappa结果
    save_kappa_results(kappa_train, kappa_test, output_dir)
    
    # 3. Delong检验 - 训练集
    pval_train, zstat_train, auc_diff_train = calculate_delong_matrix(results, y_train, dataset_type='Train')
    plot_delong_heatmap(pval_train, zstat_train, auc_diff_train, 
                        f'{output_dir}/09_delong_test_heatmap_train', dataset_type='Train')
    
    # 4. Delong检验 - 测试集
    pval_test, zstat_test, auc_diff_test = calculate_delong_matrix(results, y_test, dataset_type='Test')
    plot_delong_heatmap(pval_test, zstat_test, auc_diff_test, 
                        f'{output_dir}/09_delong_test_heatmap_test', dataset_type='Test')
    
    # 保存Delong结果
    save_delong_results(pval_train, pval_test, zstat_train, zstat_test,
                        auc_diff_train, auc_diff_test, output_dir)
    
    # 5. 校准曲线详细分析 - 训练集和测试集
    cal_train_df, cal_test_df = save_calibration_results(results, y_train, y_test, output_dir)
    
    # 6. 决策曲线分析 (DCA) - 训练集和测试集
    dca_train_df, dca_test_df = save_dca_results(results, y_train, y_test, output_dir)
    
    print("\n" + "=" * 70)
    print("模型间比较分析完成!")
    print("=" * 70)
    print(f"\n生成的新文件:")
    print(f"  热图:")
    print(f"    - 08_cohen_kappa_heatmap_train.png/pdf")
    print(f"    - 08_cohen_kappa_heatmap_test.png/pdf")
    print(f"    - 09_delong_test_heatmap_train.png/pdf")
    print(f"    - 09_delong_test_heatmap_test.png/pdf")
    print(f"  数据表:")
    print(f"    - cohen_kappa_matrix_train/test.csv/xlsx")
    print(f"    - delong_pvalues_train/test.csv/xlsx")
    print(f"    - delong_zstatistics_train/test.csv")
    print(f"    - delong_auc_diff_train/test.csv")
    print(f"  校准曲线分析:")
    print(f"    - calibration_metrics_train/test.csv/xlsx")
    print(f"  决策曲线分析 (DCA):")
    print(f"    - 10_decision_curve_train.png/pdf")
    print(f"    - 10_decision_curve_test.png/pdf")
    print(f"    - dca_metrics_train/test.csv/xlsx")



In [3]:

# ============================================================
# 主函数
# ============================================================

def main():
    """
    主执行函数
    """
    OUTPUT_DIR = './thyroid_cancer_9models_results_pureNoStacking_335PTCpatients_lightGBM'

    print(f"\n输出目录: {OUTPUT_DIR}")

    # 步骤1: 加载数据
    df = load_and_preprocess_data('./modeldata_335_PTC.csv')

    # 步骤2: 特征工程和数据保存（同时保存CSV）
    X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled, feature_names, scaler = \
        prepare_features_and_save(df, OUTPUT_DIR)

    # 步骤3-4: 定义和训练模型
    models = get_models_and_params()
    results = train_and_optimize_models(models, X_train, X_test, y_train, y_test,
                                        X_train_scaled, X_test_scaled)

    # # 步骤5: 堆叠集成    #如果不需要堆叠集成 可以注释取消
    # results = build_stacking_model(results, X_train, X_test, y_train, y_test,
    #                                X_train_scaled, X_test_scaled)

    # 步骤6: 可视化（PNG + PDF）- 支持训练集和测试集
    create_visualizations(results, X_train, X_test, y_train, y_test, feature_names, OUTPUT_DIR)

    # 步骤7: 保存结果 - 保存训练集和测试集预测结果
    metrics_df, metrics_train_df, ranking = save_results(results, X_train, X_test, y_train, y_test, OUTPUT_DIR)

    # 步骤8: SHAP分析
    shap_analysis(results, X_train, X_test, feature_names, OUTPUT_DIR)

    # 步骤9: 模型间比较分析 (Cohen Kappa + Delong Test)
    create_model_comparison_analysis(results, y_train, y_test, OUTPUT_DIR)

    # 总结
    print("\n" + "=" * 70)
    print("分析完成!")
    print("=" * 70)
    print(f"\n🏆 最佳模型: {ranking.index[0]}")
    print(f"   AUC: {ranking.iloc[0]['AUC']:.4f}")
    print(f"   F1:  {ranking.iloc[0]['F1-score']:.4f}")
    print(f"\n📁 所有结果: {OUTPUT_DIR}/")
    print("\n生成的文件包括:")
    print("  - 数据集CSV文件 (5个)")
    print("  - 模型文件 (PKL)")
    print("  - 结果表格 (CSV/Excel)")
    print("  - 可视化图表 (PNG + PDF)")
    print("\n新增分析结果:")
    print("  - Cohen's Kappa系数矩阵 (训练集/测试集)")
    print("    * cohen_kappa_matrix_train/test.csv/xlsx")
    print("    * 08_cohen_kappa_heatmap_train/test.png/pdf")
    print("  - Delong检验结果 (训练集/测试集)")
    print("    * delong_pvalues_train/test.csv/xlsx")
    print("    * delong_zstatistics_train/test.csv")
    print("    * delong_auc_diff_train/test.csv")
    print("    * 09_delong_test_heatmap_train/test.png/pdf")
    print("  - 校准曲线详细分析 (训练集/测试集)")
    print("    * calibration_metrics_train/test.csv/xlsx")
    print("    * 包含ECE, MCE, Brier Score等指标")
    print("  - 决策曲线分析 DCA (训练集/测试集)")
    print("    * 10_decision_curve_train/test.png/pdf")
    print("    * dca_metrics_train/test.csv/xlsx")
    print("    * 包含最大净收益、临床实用性指标")

if __name__ == "__main__":
    main()


输出目录: ./thyroid_cancer_9models_results_pureNoStacking_335PTCpatients_lightGBM

步骤 1: 数据加载和预处理
从 ./modeldata_335_PTC.csv 加载数据
总样本数: 335
特征数: 16

类别分布:
Recurred
0    245
1     90
Name: count, dtype: int64

复发率: 26.87%

步骤 2: 特征工程和数据集保存
特征矩阵形状: (335, 16)

训练集: 234 样本 (复发: 63, 未复发: 171)
测试集: 101 样本 (复发: 27, 未复发: 74)

✓ 保存完整数据集: complete_dataset.csv (335 样本)
✓ 保存训练集: training_set.csv (234 样本)
✓ 保存测试集: testing_set.csv (101 样本)
✓ 保存标准化训练集: training_set_scaled.csv
✓ 保存标准化测试集: testing_set_scaled.csv
✓ 保存数据集信息: dataset_info.txt
✓ 保存标准化器: scaler.pkl

步骤 3: 模型定义
共定义 9 个模型:
  - Logistic Regression
  - Decision Tree
  - SVM
  - KNN
  - Naive Bayes
  - Random Forest
  - XGBoost
  - LightGBM
  - CatBoost

步骤 4: 模型训练和超参数优化

训练模型: Logistic Regression
使用标准化数据
执行GridSearchCV...
最优参数: {'C': 0.1, 'penalty': 'l1', 'solver': 'saga'}
最优CV AUC: 0.9849

训练集性能:
  Accuracy: 0.9359 (95% CI: 0.902-0.966)
  Precision: 0.8750 (95% CI: 0.788-0.952)
  Recall: 0.8889 (95% CI: 0.805-0.958)
  F1-score: 0.8819 (95% CI: 0.8

# 🚀 Part 10: Streamlit Web App Deployment

After training, the best model (**LightGBM**) is saved as `BEST_MODEL_LightGBM.pkl`.
An interactive web application `app.py` is provided using **Streamlit**:

| Page | Description |
|------|-------------|
| 🔮 **Single Prediction** | Enter 16 clinical features via a form, get real-time recurrence probability |
| 📊 **Batch Prediction**  | Upload a CSV file for batch prediction, download the results |
| 📈 **Model Performance** | Compare 9 models and view 11 training visualizations |
| ℹ️ **About**             | Project background, modeling workflow, feature descriptions |

## Quick Launch

Run one of the following in the project directory:

**Windows (recommended):**
```
Double-click run_app.bat
```

**Command line:**
```bash
pip install -r requirements.txt
streamlit run app.py
```

The browser will open <http://localhost:8501> automatically.

## Files

- `app.py` - Streamlit main program (integrates the best LightGBM model)
- `requirements.txt` - Python dependency list
- `run_app.bat` - Windows one-click launcher
- `README_APP.md` - Detailed deployment and usage guide

In [4]:
# ============================================================
# Verify that all Streamlit deployment files are in place
# ============================================================
import os

MODEL_DIR = './thyroid_cancer_9models_results_pureNoStacking_335PTCpatients_lightGBM'

required_files = {
    'app.py': './app.py',
    'requirements.txt': './requirements.txt',
    'run_app.bat': './run_app.bat',
    'README_APP.md': './README_APP.md',
    'BEST_MODEL_LightGBM.pkl': f'{MODEL_DIR}/BEST_MODEL_LightGBM.pkl',
    'scaler.pkl': f'{MODEL_DIR}/scaler.pkl',
    'model_ranking.csv': f'{MODEL_DIR}/model_ranking.csv',
}

print("=" * 60)
print("Streamlit deployment file check")
print("=" * 60)
for name, path in required_files.items():
    exists = os.path.exists(path)
    size = f"{os.path.getsize(path)/1024:.1f} KB" if exists else "---"
    mark = "[OK]" if exists else "[!!]"
    print(f"  {mark} {name:35s} {size:>12s}  ({path})")

print("\n" + "=" * 60)
print("How to launch:")
print("=" * 60)
print("  Option 1 (Windows): double-click run_app.bat")
print("  Option 2 (CLI):     streamlit run app.py")
print("  Default URL:        http://localhost:8501")
print("=" * 60)


Streamlit deployment file check
  [!!] app.py                                       ---  (./app.py)
  [!!] requirements.txt                             ---  (./requirements.txt)
  [!!] run_app.bat                                  ---  (./run_app.bat)
  [!!] README_APP.md                                ---  (./README_APP.md)
  [OK] BEST_MODEL_LightGBM.pkl                 210.4 KB  (./thyroid_cancer_9models_results_pureNoStacking_335PTCpatients_lightGBM/BEST_MODEL_LightGBM.pkl)
  [OK] scaler.pkl                                1.5 KB  (./thyroid_cancer_9models_results_pureNoStacking_335PTCpatients_lightGBM/scaler.pkl)
  [OK] model_ranking.csv                         3.2 KB  (./thyroid_cancer_9models_results_pureNoStacking_335PTCpatients_lightGBM/model_ranking.csv)

How to launch:
  Option 1 (Windows): double-click run_app.bat
  Option 2 (CLI):     streamlit run app.py
  Default URL:        http://localhost:8501
